<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 3 — dbt practical and consolidation

<small>Data Creators › Day 3 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

### Ice-breaker — one thing you always double-check

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 ICE-BREAKER — one thing you always double-check</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">What is one thing you always double-check, even though it is almost always fine? One answer each, at work or at home.</div></div>

- The front door. The oven. The attachment before you press send.
- The recipient of an email. The total on a bill. The alarm for tomorrow.

**Answers from the room:**

- Gas knobs at stove -- ALWAYS DOUBLE CHECK
- Gates on the front yard for the docs
- Emails 
- Keys, phone, wallets
- Cat food
- bagdes
- lights
- chores for your son

<a id="how-to-use"></a>

## About this notebook

- Day 3's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [47]:
# Setup: the demo environment on PATH, a sql() helper for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
# The notebook's own folder, courseware/day3-*/, found from wherever the kernel is now, so this cell can be
# run again after a %cd into a demo folder. It also changes back to that folder.
HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "day3_anchor.ipynb").exists()), None)
assert HERE is not None, "Open this notebook from its own folder (courseware/day3-*)"
os.chdir(HERE)
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day3-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a demo warehouse file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 3 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

✅ Day 3 setup: - installed: 1.11.8


In [48]:
# The two demo warehouses the cells below query: local files on the instructor's laptop
PRICE_SHEET = HERE / "demos/price_sheet/warehouse/price_sheet.duckdb"
PORTFOLIO = HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb"

## Today

A short recap of yesterday's platform, forty minutes of dbt from zero on a small example, then self-paced practice from the README, with the instructor walking the room, all the way to lunch. After lunch, twenty-five minutes of advanced dbt, then more practice. Recap and a pair exercise at the end.

| # | Section | What happens |
|---|---|---|
| 1 | **[dbt from zero: one transformation through the whole life cycle](#s1)** | Recap: yesterday's platform in two pictures · Demo: what dbt is for, why use it, then the build: project, profile, seeds, source, model, Jinja, ref, tests, build, docs |
| 2 | **[The dbt practical: launch, Phase 1 and the walkthrough](#s2)** | Launch: the pipeline · Challenge 9: dbt practical · Walkthrough: Phase 1 · Break · Walkthrough: Phase 2 · Circulating: Phase 2 to lunch · Lunch |
| 3 | **[Advanced dbt, then complete and extend the practical](#s3)** | Demo: a loop, a macro, a custom test, snapshot, incremental, OPTIMIZE and ZORDER, unit test · Challenge 9 continued: Phase 2 and the stretch · Break |
| 4 | **[Consolidation](#s4)** | Activity 9b: second look at yesterday's AI-generated dbt · Debrief: what did we learn? · Activity 9: test review in pairs · Close |

#### TODO: Guest at 1 PM 

<a id="s1"></a>

## 1. dbt from zero: one transformation through the whole life cycle

### Recap — yesterday's platform in two pictures

Ten minutes to put everyone on the same page. Today's dbt models are SQL that runs on this platform; one question per picture.

<img src="../common/diagrams/png/d3-recap-lakehouse.png" alt="Data warehouse, data lake, lakehouse" style="max-height:400px; max-width:100%;">

**Think:** What does a lakehouse keep from the lake, and what does it add?

**Answers from the room:**

- Combinations of functionalities
- ACID compliance
- Governance via Unity cataloug

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Cheap files in object storage, plus a table format (Delta) for transactions and versions, an engine (Spark) for SQL, and a catalog (Unity Catalog) for names and permissions.
- **Today:** Every dbt model becomes a table or a view in that lakehouse.

</details>

![Databricks under the hood: sources, ingestion, Delta storage, Spark compute, Unity Catalog, consumers](../common/diagrams/png/d3-recap-databricks.png)

**Think:** Where is your data stored, and what runs your SQL?

**Answers from the room:**

- Parquet + delta logs - storage format 
- Answer D, S, C aligned Q
- Lakehouse publishing back to Unity catalog
- Is it difference from workspace to an Enterprise 

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Storage is your S3 bucket in Delta format. Compute is Spark, on a SQL warehouse. Unity Catalog names every table `catalog.schema.table`.
- **Today:** Your tables land in `training_dev.training_<you>`; the raw data arrives with `dbt seed`.

</details>

### Demo — one price sheet, from two raw files to a tested, documented table

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday Catalyst generated a dbt project for you. What is in it, and what does dbt do with each file?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>dbt on the smallest useful example: two raw files, four models in three layers, three tests, one documented table, on Day 2's price data.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Every file in today's challenge is one of the kinds shown here. Once you have seen each one written and run, the README's tasks are recognisable.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Watch and predict: each cell on screen writes one file or runs one command. You do not run these cells. In forty minutes you build the real pipeline yourself, on Databricks.</div></div></div>

#### First: what dbt is for

![dbt is the T in extract, load, transform: SQL files in, tables and views out](../common/diagrams/png/d3-what-dbt-does.png)

*dbt is the T in extract, load, transform: SQL files in, tables and views out*

<details><summary>Notes</summary>

- Other tools load the raw data. You write SELECT statements in files, kept in Git; dbt wraps each in a CREATE, runs them in order and tests the results.
- dbt holds no data and does no computing: the warehouse does the work.
- Analysts, analytics engineers and data engineers use the same files. At Capital Group every data product on the paved path has a dbt project; Airflow triggers a Databricks job that runs it.

</details>

#### Why use it: the same price sheet without dbt

In [49]:
# Without dbt: one SQL script, run by hand. Prices in US dollars = close price x the day's FX rate.
seeds = HERE / "demos/price_sheet/seeds"
sql(f"""
    select p.isin, p.px_date, p.px_ccy, p.px_close,
           round(p.px_close * coalesce(f.rate, 1), 4) as price_usd
    from '{seeds}/vendor_prices_raw.csv' p
    left join '{seeds}/fx_rates_raw.csv' f
      on f.from_ccy = p.px_ccy and f.rate_dt = p.px_date
    where p.px_ccy <> 'USD'
    order by p.isin, p.px_date
""", ":memory:", read_only=False)
# It ran, with no error. Now look at HSBC (GB0005405286): the dollar price equals the pound price. Who would have noticed?

,isin,px_date,px_ccy,px_close,price_usd
0,DE0007164600,2026-11-03,EUR,214.1301,231.5943
1,DE0007164600,2026-11-04,EUR,210.8061,227.8264
2,GB0005405286,2026-11-03,gbp,6.9883,6.9883
3,GB0005405286,2026-11-04,gbp,7.2556,7.2556
4,GB0005405286,2026-11-05,gbp,7.1336,7.1336
5,JP3633400001,2026-11-03,JPY,2697.8000,17.7947
6,JP3633400001,2026-11-04,JPY,2784.4000,18.3715
7,JP3633400001,2026-11-05,JPY,2697.0000,17.8299


The script ran, and three prices are wrong. The SQL is the easy part; dbt is for everything around it.

<img src="../common/diagrams/png/d3-script-vs-dbt.png" alt="Six questions a hand-run script cannot answer, and what dbt gives you for each" style="max-height:360px; max-width:100%;">

*Six questions a hand-run script cannot answer, and what dbt gives you for each*

#### How a dbt project comes together: the path for the next thirty minutes

![The path: eight steps from an empty folder to a tested, documented table](../common/diagrams/png/d3-journey.png)

*The path: eight steps from an empty folder to a tested, documented table*

<details><summary>Notes</summary>

- Each step adds one kind of file and one command. First make it run (steps 1 to 4), then make it right and say what it is (steps 5 to 8).
- It is the order you would work in on a real product, and the order of the challenge README.

</details>

#### The layers: staging, intermediate, marts

![Every dbt project here has the same three layers. Each has one job, a folder and a prefix.](../common/diagrams/png/d3-layers.png)

*Every dbt project here has the same three layers. Each has one job, a folder and a prefix.*

<details><summary>Notes</summary>

- **staging/** (`stg_`): one model per raw table. Rename, cast, clean. No joins. Views.
- **intermediate/** (`int_`): the joins and the business rules. Views; nobody outside the project reads them.
- **marts/** (`fct_`, `agg_`): the tables consumers read. Tables, because they are queried often.
- The challenge has the same folders and prefixes: five `stg_`, three `int_`, three marts. Catalyst and the capstone use the names source-aligned, domain-aligned and consumer-aligned for the same idea.

</details>

#### What we will build: four models in three layers

<img src="../common/diagrams/png/d3-build-plan.png" alt="The plan. Dashed boxes are not built yet; this picture comes back after each step with the new piece highlighted." style="max-height:300px; max-width:100%;">

*The plan. Dashed boxes are not built yet; this picture comes back after each step with the new piece highlighted.*

<details><summary>Notes</summary>

- Two raw tables, two staging models, one intermediate model that joins them, one mart.
- Yellow: built in this step. Blue: built earlier. Red: a test is failing.

</details>

#### Step 1 — A dbt project is a folder: two YAML files and a folder of SQL

![The anatomy of a dbt project folder](../common/diagrams/png/d3-project-anatomy.png)

*The anatomy of a dbt project folder*

<details><summary>Notes</summary>

- `dbt_project.yml`: the project's name and its defaults. `profiles.yml`: the connection, one entry per target (dev, prod).
- `models/`: one SELECT per `.sql` file, in three folders, plus YAML that describes and tests them. `staging/` cleans one raw table per model (`stg_`), `intermediate/` joins and applies the rules (`int_`), `marts/` holds the tables consumers read (`fct_`, `agg_`). The challenge uses the same folders and prefixes.
- `seeds/`: small CSVs dbt loads. `tests/` and `macros/`: later this morning and after lunch.
- `target/`: what dbt writes when it runs (compiled SQL, run results, the docs site). Never edited, never committed.

</details>

In [50]:
%cd {HERE}/demos/price_sheet
# Work inside the project folder from here on. dbt commands run from the folder that holds dbt_project.yml; in the challenge that is challenges/v1/02-dbt-pipeline/dbt.

/Users/usama.shahid/Library/CloudStorage/GoogleDrive-usama.shahid@decoded.com/My Drive/cg-dxp/courseware/day3-dbt/demos/price_sheet


In [51]:
!bash reset.sh
# The starting point: two raw CSV files and nothing else. `dbt init` would scaffold the folder; here every file is written by hand so you see what is in it.

price_sheet/ is back to its starting point:
./README.md
./reset.sh
./seeds/fx_rates_raw.csv
./seeds/vendor_prices_raw.csv


In [52]:
%%writefile dbt_project.yml
# dbt_project.yml: what this project is called and how each folder of models is built
name: price_sheet
version: '1.0'
profile: price_sheet          # which connection in profiles.yml to use

models:
  price_sheet:
    staging:                  # models/staging/: one model per raw table, cleaned
      +materialized: view
      +tags: ["staging"]
    intermediate:             # models/intermediate/: the joins and the business rules
      +materialized: view
      +tags: ["intermediate"]
    marts:                    # models/marts/: the tables consumers read
      +materialized: table
      +tags: ["marts"]

Writing dbt_project.yml


In [53]:
%%writefile profiles.yml
# profiles.yml: where the SQL runs. One project, one or more targets.
price_sheet:
  target: dev
  outputs:
    dev:
      type: duckdb                          # the demo's warehouse: a local file. Yours says: type: databricks
      path: warehouse/price_sheet.duckdb    # ...and yours has: host, http_path, catalog
      schema: demo                          # the schema the models are created in

Writing profiles.yml


In [54]:
!dbt debug
# dbt debug answers one question: can dbt find the project, find the profile and open the connection? It is the first command in the challenge too.

20:53:11  Running with dbt=1.11.8
20:53:11  dbt version: 1.11.8
20:53:11  python version: 3.13.5
20:53:11  python path: /Users/usama.shahid/.venvs/cg-dxp/bin/python
20:53:11  os info: macOS-27.2-arm64-arm-64bit-Mach-O
20:53:11  Using profiles dir at /Users/usama.shahid/Library/CloudStorage/GoogleDrive-usama.shahid@decoded.com/My Drive/cg-dxp/courseware/day3-dbt/demos/price_sheet
20:53:11  Using profiles.yml file at /Users/usama.shahid/Library/CloudStorage/GoogleDrive-usama.shahid@decoded.com/My Drive/cg-dxp/courseware/day3-dbt/demos/price_sheet/profiles.yml
20:53:11  Using dbt_project.yml file at /Users/usama.shahid/Library/CloudStorage/GoogleDrive-usama.shahid@decoded.com/My Drive/cg-dxp/courseware/day3-dbt/demos/price_sheet/dbt_project.yml
20:53:11  adapter type: duckdb
20:53:11  adapter version: 1.11.1
20:53:11  Configuration:
20:53:11    profiles.yml file [OK found and valid]
20:53:11    dbt_project.yml file [OK found and valid]
20:53:11  Required dependencies:
20:53:11   - git [OK

#### Step 2 — The raw data: seeds stand in for the feeds that land in the warehouse

In [55]:
!dbt seed

20:53:12  Running with dbt=1.11.8
20:53:12  Registered adapter: duckdb=1.11.1
20:53:12  Unable to do partial parsing because saved manifest not found. Starting full parse.
20:53:13  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 3 unused configuration paths:
- models.price_sheet.intermediate
- models.price_sheet.marts
- models.price_sheet.staging
20:53:13  Found 2 seeds, 511 macros
20:53:13  
20:53:13  Concurrency: 1 threads (target='dev')
20:53:13  
20:53:13  1 of 2 START seed file demo.fx_rates_raw ....................................... [RUN]
20:53:13  1 of 2 OK loaded seed file demo.fx_rates_raw ................................... [INSERT 9 in 0.03s]
20:53:13  2 of 2 START seed file demo.vendor_prices_raw .................................. [RUN]
20:53:13  2 of 2 OK loaded seed file demo.vendor_prices_raw .............................. [INSERT 36 in 0.01s]
20:53:13  
20:53:13  Finished running 2 seeds in 0 hours 0 min

In [56]:
!dbt show --inline "select isin, px_date, px_close, px_ccy from demo.vendor_prices_raw where px_ccy <> 'USD' order by isin, px_date" --limit 10
# dbt show --inline runs any query in the warehouse and prints the rows. These are the prices that are not in US dollars. Look closely at the currency of the second instrument: gbp, in lower case. It matters in a few minutes.

20:53:14  Running with dbt=1.11.8
20:53:14  Registered adapter: duckdb=1.11.1
20:53:14  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 3 unused configuration paths:
- models.price_sheet.staging
- models.price_sheet.intermediate
- models.price_sheet.marts
20:53:14  Found 2 seeds, 1 sql operation, 511 macros
20:53:14  
20:53:14  Concurrency: 1 threads (target='dev')
20:53:14  
Previewing inline node:
| isin         |    px_date |   px_close | px_ccy |
| ------------ | ---------- | ---------- | ------ |
| DE0007164600 | 2026-11-03 |   214.130… | EUR    |
| DE0007164600 | 2026-11-04 |   210.806… | EUR    |
| GB0005405286 | 2026-11-03 |     6.988… | gbp    |
| GB0005405286 | 2026-11-04 |     7.256… | gbp    |
| GB0005405286 | 2026-11-05 |     7.134… | gbp    |
| JP3633400001 | 2026-11-03 | 2,697.800… | JPY    |
| JP3633400001 | 2026-11-04 | 2,784.400… | JPY    |
| JP3633400001 | 2026-11-05 | 2,697.000… | JPY    |



In [57]:
!dbt show --select fx_rates_raw --limit 10
# One rate per currency per day, to US dollars. The transformation: price in dollars = close price x that day's rate.

20:53:15  Running with dbt=1.11.8
20:53:15  Registered adapter: duckdb=1.11.1
20:53:15  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 3 unused configuration paths:
- models.price_sheet.marts
- models.price_sheet.intermediate
- models.price_sheet.staging
20:53:16  Found 2 seeds, 511 macros
20:53:16  
20:53:16  Concurrency: 1 threads (target='dev')
20:53:16  
20:53:16  1 of 1 START seed file demo.fx_rates_raw ....................................... [RUN]
20:53:16  1 of 1 OK loaded seed file demo.fx_rates_raw ................................... [INSERT 9 in 0.03s]
Previewing node 'fx_rates_raw':
| from_ccy | to_ccy |    rate_dt |   rate |        _received_at |
| -------- | ------ | ---------- | ------ | ------------------- |
| GBP      | USD    | 2026-11-03 | 1.267… | 2026-11-03 17:00:00 |
| EUR      | USD    | 2026-11-03 | 1.082… | 2026-11-03 17:00:00 |
| JPY      | USD    | 2026-11-03 | 0.007… | 2026-11-03 17:00:00 |
| G

<img src="../common/diagrams/png/d3-build-2.png" alt="Where we are: the raw data is loaded. No models yet." style="max-height:300px; max-width:100%;">

*Where we are: the raw data is loaded. No models yet.*

#### Step 3 — Declare the source, write the first model, run it

<img src="../common/diagrams/png/d3-seed-vs-source.png" alt="Seeds stand in for the feeds: however a raw table arrives, you declare it in sources.yml and read it with source()" style="max-height:300px; max-width:100%;">

*Seeds stand in for the feeds: however a raw table arrives, you declare it in sources.yml and read it with source()*

<details><summary>Notes</summary>

- Here and in the challenge, `dbt seed` loads the CSVs. We then declare those tables under `sources:` and read them with `source(...)`, as if another tool had landed them.
- That is how raw data arrives on a real product: an ingestion tool lands it, and dbt only declares it and reads it.
- So the models you write today would not change on a real product; only where the raw tables come from.

</details>

In [58]:
%%writefile models/sources.yml
# sources.yml: the raw tables this project reads but does not build
version: 2

sources:
  - name: vendor_feed                 # a label you choose for a group of raw tables, usually the system they come from.
                                      # Not a database: it is the first word in source('vendor_feed', 'vendor_prices_raw')
    description: "End-of-day close prices from the pricing vendor, in each instrument's own currency."
    schema: demo                      # where the raw tables really live in the warehouse
                                      # (on Databricks you can add  database: <catalog>  too; left out, dbt uses your profile's)
    tables:
      - name: vendor_prices_raw       # the real table name: the second word in source(...)

  - name: fx_feed                     # a second group: one source per system
    description: "Daily FX closes to US dollars."
    schema: demo
    tables:
      - name: fx_rates_raw

Writing models/sources.yml


![YAML files in dbt: .yml and .yaml are the same; only two names are fixed. Today every file is .yml, as in the challenge.](../common/diagrams/png/d3-yaml.png)

*YAML files in dbt: .yml and .yaml are the same; only two names are fixed. Today every file is .yml, as in the challenge.*

<details><summary>Notes</summary>

- `dbt_project.yml` and `profiles.yml` must have exactly those names.
- Every other YAML file can have any name: dbt reads the top-level keys inside (`sources:`, `models:`, `seeds:`, `unit_tests:`), not the file name. `sources.yml` and `schema.yml` are habits; Catalyst writes `_staging__sources.yml` and `_staging__models.yml`.
- You will see `.yaml` in other tools this week: Airflow blueprints, Databricks bundles, data contracts.
- Indentation is the syntax: two spaces per level, never tabs.

</details>

In [59]:
%%writefile models/staging/stg_prices.sql
-- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    isin                               as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    px_ccy                             as native_ccy,
    cast(vendor_ts as timestamp)       as vendor_ts
from {{ source('vendor_feed', 'vendor_prices_raw') }}

Writing models/staging/stg_prices.sql


![What dbt run does with a model file: compile, wrap, send](../common/diagrams/png/d3-compile.png)

*What dbt run does with a model file: compile, wrap, send*

<details><summary>Notes</summary>

- Compile: the curly braces are replaced, here `source(...)` becomes the real table name. The result is plain SQL in `target/compiled/`.
- Wrap: dbt puts your SELECT inside `create view ... as` or `create table ... as`, according to the model's materialisation. That is in `target/run/`.
- Send: the warehouse executes it. The model's file name is the name of the view or table.

</details>

In [60]:
!dbt run

20:53:17  Running with dbt=1.11.8
20:53:17  Registered adapter: duckdb=1.11.1
20:53:17  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.price_sheet.marts
- models.price_sheet.intermediate
20:53:17  Found 2 seeds, 1 model, 2 sources, 511 macros
20:53:17  
20:53:17  Concurrency: 1 threads (target='dev')
20:53:17  
20:53:17  1 of 1 START sql view model demo.stg_prices .................................... [RUN]
20:53:17  1 of 1 OK created sql view model demo.stg_prices ............................... [OK in 0.03s]
20:53:17  
20:53:17  Finished running 1 view model in 0 hours 0 minutes and 0.11 seconds (0.11s).
20:53:17  
20:53:17  Completed successfully
20:53:17  
20:53:17  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


In [61]:
!dbt show --select stg_prices --limit 6
# dbt show runs the query of a model and prints the first rows: the way to look at a result without leaving dbt. One model, one view called stg_prices. You wrote a SELECT; dbt wrote the CREATE.

20:53:18  Running with dbt=1.11.8
20:53:18  Registered adapter: duckdb=1.11.1
20:53:18  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.price_sheet.marts
- models.price_sheet.intermediate
20:53:19  Found 2 seeds, 1 model, 2 sources, 511 macros
20:53:19  
20:53:19  Concurrency: 1 threads (target='dev')
20:53:19  
Previewing node 'stg_prices':
| instrument_id | price_date | price_native | native_ccy |           vendor_ts |
| ------------- | ---------- | ------------ | ---------- | ------------------- |
| US0378331005  | 2026-11-03 |     229.869… | USD        | 2026-11-03 21:05:00 |
| US5949181045  | 2026-11-03 |     422.121… | USD        | 2026-11-03 21:05:00 |
| US0231351067  | 2026-11-03 |     202.415… | USD        | 2026-11-03 21:05:00 |
| US30303M1027  | 2026-11-03 |     580.303… | USD        | 2026-11-03 21:05:00 |
| US67066G1040  | 2026-11-03 |     140.001… | USD        | 2026-11

<img src="../common/diagrams/png/d3-build-3.png" alt="Where we are: the first staging model is built. Three models to go." style="max-height:300px; max-width:100%;">

*Where we are: the first staging model is built. Three models to go.*

In [62]:
!cat target/compiled/price_sheet/models/staging/stg_prices.sql
# What dbt compiled: your SELECT, with the curly braces replaced by the real table name.

-- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    isin                               as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    px_ccy                             as native_ccy,
    cast(vendor_ts as timestamp)       as vendor_ts
from "price_sheet"."demo"."vendor_prices_raw"

In [63]:
!cat target/run/price_sheet/models/staging/stg_prices.sql
# What the warehouse ran: the same SELECT, wrapped in create view ... as.


  
  create view "price_sheet"."demo"."stg_prices__dbt_tmp" as (
    -- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    isin                               as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    px_ccy                             as native_ccy,
    cast(vendor_ts as timestamp)       as vendor_ts
from "price_sheet"."demo"."vendor_prices_raw"
  );


<img src="../common/diagrams/png/d3-jinja.png" alt="Jinja, the curly braces: dbt replaces them before the warehouse sees the SQL" style="max-height:460px; max-width:100%;">

*Jinja, the curly braces: dbt replaces them before the warehouse sees the SQL*

<details><summary>Notes</summary>

- Two curly braces print a value into the SQL. In the challenge: `source()` from S4, `config()` in S6, `ref()` from C1, `this` in C9.
- A curly brace and a percent sign run logic and print nothing: `if`, `for`, `macro`. Not needed in the challenge; at 13:00.
- Jinja runs on your machine at compile time. When a model misbehaves, read its file under `target/compiled/`.

</details>

#### Step 4 — The other layers with ref(): staging, intermediate, marts, and dbt works out the order

In [64]:
%%writefile models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

Writing models/staging/stg_fx_rates.sql


In [65]:
%%writefile models/intermediate/int_prices_usd.sql
-- int_prices_usd: every price joined to its day's FX rate and converted. Same rows as stg_prices.
select
    p.instrument_id,
    p.price_date,
    p.native_ccy,
    p.price_native,
    case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end   as fx_rate_to_usd,
    cast(p.price_native * case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end
         as decimal(18, 6))                                      as price_usd
from {{ ref('stg_prices') }} p
left join {{ ref('stg_fx_rates') }} f
    on  f.from_ccy  = p.native_ccy
    and f.rate_date = p.price_date

Writing models/intermediate/int_prices_usd.sql


In [66]:
%%writefile models/marts/fct_prices_usd.sql
-- fct_prices_usd: the price sheet consumers read. One row per instrument per day.
select
    instrument_id,
    price_date,
    native_ccy,
    price_native,
    fx_rate_to_usd,
    price_usd
from {{ ref('int_prices_usd') }}

Writing models/marts/fct_prices_usd.sql


In [67]:
!dbt run
# Four models, built in graph order: the two staging models, then the intermediate one, then the mart. Read the log: three say "view model", one says "table model".

20:53:20  Running with dbt=1.11.8
20:53:20  Registered adapter: duckdb=1.11.1
20:53:21  Found 2 seeds, 4 models, 2 sources, 511 macros
20:53:21  
20:53:21  Concurrency: 1 threads (target='dev')
20:53:21  
20:53:21  1 of 4 START sql view model demo.stg_fx_rates .................................. [RUN]
20:53:21  1 of 4 OK created sql view model demo.stg_fx_rates ............................. [OK in 0.03s]
20:53:21  2 of 4 START sql view model demo.stg_prices .................................... [RUN]
20:53:21  2 of 4 OK created sql view model demo.stg_prices ............................... [OK in 0.01s]
20:53:21  3 of 4 START sql view model demo.int_prices_usd ................................ [RUN]
20:53:21  3 of 4 OK created sql view model demo.int_prices_usd ........................... [OK in 0.01s]
20:53:21  4 of 4 START sql table model demo.fct_prices_usd ............................... [RUN]
20:53:21  4 of 4 OK created sql table model demo.fct_prices_usd .......................... [

<img src="../common/diagrams/png/d3-build-4.png" alt="Where we are: all four models built, in the order dbt worked out from source() and ref()" style="max-height:300px; max-width:100%;">

*Where we are: all four models built, in the order dbt worked out from source() and ref()*

<details><summary>Notes</summary>

- Every `source()` and `ref()` is an edge. dbt reads them all, then builds the models from left to right.
- You never write the order down. Add a model and the graph changes by itself.
- The same graph is the lineage picture in the docs, and what `--select +fct_prices_usd` walks.

</details>

In [68]:
!dbt show --inline "select instrument_id, price_date, native_ccy, price_native, fx_rate_to_usd, price_usd from demo.fct_prices_usd where native_ccy <> 'USD' order by instrument_id, price_date" --limit 10
# The run was green. Look at the HSBC rows (GB0005405286): no rate, no dollar price. A successful run says the SQL executed, not that the answer is right.

20:53:22  Running with dbt=1.11.8
20:53:22  Registered adapter: duckdb=1.11.1
20:53:22  Found 2 seeds, 4 models, 1 sql operation, 2 sources, 511 macros
20:53:22  
20:53:22  Concurrency: 1 threads (target='dev')
20:53:22  
Previewing inline node:
| instrument_id | price_date | native_ccy | price_native | fx_rate_to_usd | price_usd |
| ------------- | ---------- | ---------- | ------------ | -------------- | --------- |
| DE0007164600  | 2026-11-03 | EUR        |     214.130… |         1.082… |  231.594… |
| DE0007164600  | 2026-11-04 | EUR        |     210.806… |         1.081… |  227.826… |
| GB0005405286  | 2026-11-03 | gbp        |       6.988… |                |           |
| GB0005405286  | 2026-11-04 | gbp        |       7.256… |                |           |
| GB0005405286  | 2026-11-05 | gbp        |       7.134… |                |           |
| JP3633400001  | 2026-11-03 | JPY        |   2,697.800… |         0.007… |   17.795… |
| JP3633400001  | 2026-11-04 | JPY        |   2,78

#### Step 4, continued — View or table? Set per folder in dbt_project.yml, overridden in one model with config()

![A materialisation is how dbt saves a model: view, table, incremental or snapshot](../common/diagrams/png/d3-materialisations.png)

*A materialisation is how dbt saves a model: view, table, incremental or snapshot*

<details><summary>Notes</summary>

- View: a saved query, built on read. Cheap and always fresh. Staging and intermediate, because nobody reads them directly.
- Table: written when dbt runs. Fast to read. Marts, because consumers and dashboards read them.
- Set once per folder in `dbt_project.yml`; `config()` at the top of a model file overrides it for that model.
- Incremental: a big table where each run adds only the new rows, so it does not rebuild everything. Snapshot: a table that keeps each old version of a row with the dates it was true, so you can see what a value used to be. Both are shown at 13:00; today you only need view and table.
- In the challenge: S6, and the `models:` block you read in S2.

</details>

In [69]:
!cat dbt_project.yml

# dbt_project.yml: what this project is called and how each folder of models is built
name: price_sheet
version: '1.0'
profile: price_sheet          # which connection in profiles.yml to use

models:
  price_sheet:
    staging:                  # models/staging/: one model per raw table, cleaned
      +materialized: view
      +tags: ["staging"]
    intermediate:             # models/intermediate/: the joins and the business rules
      +materialized: view
      +tags: ["intermediate"]
    marts:                    # models/marts/: the tables consumers read
      +materialized: table
      +tags: ["marts"]


In [70]:
!dbt show --inline "select table_name, table_type from information_schema.tables where table_schema = 'demo' and table_name not like '%_raw' order by table_name" --limit 10
# The folder decides: staging/ and intermediate/ are views, marts/ is a table. Nothing in the SQL files says so.

20:53:23  Running with dbt=1.11.8
20:53:24  Registered adapter: duckdb=1.11.1
20:53:24  Found 2 seeds, 4 models, 1 sql operation, 2 sources, 511 macros
20:53:24  
20:53:24  Concurrency: 1 threads (target='dev')
20:53:24  
Previewing inline node:
| table_name     | table_type |
| -------------- | ---------- |
| fct_prices_usd | BASE TABLE |
| int_prices_usd | VIEW       |
| stg_fx_rates   | VIEW       |
| stg_prices     | VIEW       |



In [71]:
%%writefile models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
{{ config(materialized='table') }}     -- this one model: a table, whatever the folder says

select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

Overwriting models/staging/stg_fx_rates.sql


In [72]:
!dbt run --select stg_fx_rates

20:53:25  Running with dbt=1.11.8
20:53:25  Registered adapter: duckdb=1.11.1
20:53:26  Found 2 seeds, 4 models, 2 sources, 511 macros
20:53:26  
20:53:26  Concurrency: 1 threads (target='dev')
20:53:26  
20:53:26  1 of 1 START sql table model demo.stg_fx_rates ................................. [RUN]
20:53:26  1 of 1 OK created sql table model demo.stg_fx_rates ............................ [OK in 0.05s]
20:53:26  
20:53:26  Finished running 1 table model in 0 hours 0 minutes and 0.14 seconds (0.14s).
20:53:26  
20:53:26  Completed successfully
20:53:26  
20:53:26  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


In [73]:
!dbt show --inline "select table_name, table_type from information_schema.tables where table_schema = 'demo' and table_name = 'stg_fx_rates'" --limit 10
# config() in the file beats the folder setting: the log now says "table model". This is the variation in challenge S6.

20:53:27  Running with dbt=1.11.8
20:53:27  Registered adapter: duckdb=1.11.1
20:53:27  Found 2 seeds, 4 models, 1 sql operation, 2 sources, 511 macros
20:53:27  
20:53:27  Concurrency: 1 threads (target='dev')
20:53:27  
Previewing inline node:
| table_name   | table_type |
| ------------ | ---------- |
| stg_fx_rates | BASE TABLE |



In [74]:
%%writefile models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

Overwriting models/staging/stg_fx_rates.sql


In [75]:
!dbt run --select stg_fx_rates
# Back to the folder's default, a view: one line removed, nothing else changed.

20:53:28  Running with dbt=1.11.8
20:53:28  Registered adapter: duckdb=1.11.1
20:53:29  Found 2 seeds, 4 models, 2 sources, 511 macros
20:53:29  
20:53:29  Concurrency: 1 threads (target='dev')
20:53:29  
20:53:29  1 of 1 START sql view model demo.stg_fx_rates .................................. [RUN]
20:53:29  1 of 1 OK created sql view model demo.stg_fx_rates ............................. [OK in 0.03s]
20:53:29  
20:53:29  Finished running 1 view model in 0 hours 0 minutes and 0.12 seconds (0.12s).
20:53:29  
20:53:29  Completed successfully
20:53:29  
20:53:29  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


#### Step 5 — Tests: write the promise down, and dbt checks it on every run

![Two kinds of dbt test, one idea: select the rows that break the rule](../common/diagrams/png/d3-tests.png)

*Two kinds of dbt test, one idea: select the rows that break the rule*

<details><summary>Notes</summary>

- A generic test is a named rule attached to a column in YAML: `not_null`, `unique`, `accepted_values`, `relationships`.
- A singular test is a SQL file in `tests/` for a rule the four cannot express.
- Both compile to a query for the violating rows. No rows returned: pass. The same YAML holds the descriptions the docs show.

</details>

In [76]:
%%writefile models/marts/schema.yml
# schema.yml: what the model and its columns mean, and the promises dbt checks
version: 2

models:
  - name: fct_prices_usd
    description: "End-of-day prices in US dollars."
    columns:
      - name: instrument_id
        description: "ISIN: the instrument the price is for."
        tests:
          - not_null
      - name: price_usd
        description: "Close price times the day's FX rate to USD."
        tests:
          - not_null

Writing models/marts/schema.yml


In [77]:
!dbt test    # one test fails here, on purpose
# Two promises: every row has an instrument, and every row has a dollar price. The second one fails, on three rows.

20:53:30  Running with dbt=1.11.8
20:53:30  Registered adapter: duckdb=1.11.1
20:53:30  Found 2 seeds, 4 models, 2 data tests, 2 sources, 511 macros
20:53:30  
20:53:30  Concurrency: 1 threads (target='dev')
20:53:30  
20:53:30  1 of 2 START test not_null_fct_prices_usd_instrument_id ........................ [RUN]
20:53:30  1 of 2 PASS not_null_fct_prices_usd_instrument_id .............................. [PASS in 0.02s]
20:53:30  2 of 2 START test not_null_fct_prices_usd_price_usd ............................ [RUN]
20:53:30  2 of 2 FAIL 3 not_null_fct_prices_usd_price_usd ................................ [FAIL 3 in 0.01s]
20:53:30  
20:53:30  Finished running 2 data tests in 0 hours 0 minutes and 0.10 seconds (0.10s).
20:53:30  
20:53:30  Completed with 1 error, 0 partial successes, and 0 warnings:
20:53:30  
20:53:30  Failure in test not_null_fct_prices_usd_price_usd (models/marts/schema.yml)
20:53:30    Got 3 results, configured to fail if != 0
20:53:30  
20:53:30    compiled code at 

In [78]:
!cat target/compiled/price_sheet/models/marts/schema.yml/not_null_fct_prices_usd_price_usd.sql
# A test is only a query: it selects the rows that break the promise. No rows back means PASS.


    
    



select price_usd
from "price_sheet"."demo"."fct_prices_usd"
where price_usd is null




In [79]:
!dbt show --inline "select instrument_id, price_date, native_ccy, price_native, fx_rate_to_usd, price_usd from demo.fct_prices_usd where price_usd is null" --limit 10
# The three rows the test found. Their currency is written gbp, the rates table says GBP, so the join found no rate and the dollar price is empty.

20:53:31  Running with dbt=1.11.8
20:53:32  Registered adapter: duckdb=1.11.1
20:53:32  Found 2 seeds, 4 models, 2 data tests, 1 sql operation, 2 sources, 511 macros
20:53:32  
20:53:32  Concurrency: 1 threads (target='dev')
20:53:32  
Previewing inline node:
| instrument_id | price_date | native_ccy | price_native | fx_rate_to_usd | price_usd |
| ------------- | ---------- | ---------- | ------------ | -------------- | --------- |
| GB0005405286  | 2026-11-03 | gbp        |       6.988… |                |           |
| GB0005405286  | 2026-11-04 | gbp        |       7.256… |                |           |
| GB0005405286  | 2026-11-05 | gbp        |       7.134… |                |           |



<img src="../common/diagrams/png/d3-build-5.png" alt="Where we are: one test fails on the mart. The cause is upstream, in staging, where the currency should be cleaned." style="max-height:300px; max-width:100%;">

*Where we are: one test fails on the mart. The cause is upstream, in staging, where the currency should be cleaned.*

#### Step 6 — Fix the model, then dbt build: run and test together

In [80]:
%%writefile models/staging/stg_prices.sql
-- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    upper(trim(isin))                  as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    upper(trim(px_ccy))                as native_ccy,      -- the fix: 'gbp' becomes 'GBP'
    cast(vendor_ts as timestamp)       as vendor_ts
from {{ source('vendor_feed', 'vendor_prices_raw') }}

Overwriting models/staging/stg_prices.sql


![The dbt commands, and dbt build that chains them](../common/diagrams/png/d3-commands.png)

*The dbt commands, and dbt build that chains them*

<details><summary>Notes</summary>

- `dbt build` runs seeds, models and tests in graph order; if a test fails, everything downstream of it is skipped.
- `--select` narrows any command: one model, `+model` with everything upstream, `model+` with everything downstream, `tag:staging` for a layer.
- Daily loop: edit a file, `dbt build --select` that model, read the result.

</details>

Every dbt command by objective, the `--select` syntax, the files dbt reads and writes, and what to do when something goes wrong: `common/glossaries/dbt-commands.md`. Keep it open during the challenge.

In [81]:
!dbt build

20:53:33  Running with dbt=1.11.8
20:53:33  Registered adapter: duckdb=1.11.1
20:53:33  Found 2 seeds, 4 models, 2 data tests, 2 sources, 511 macros
20:53:33  
20:53:33  Concurrency: 1 threads (target='dev')
20:53:33  
20:53:33  1 of 8 START sql view model demo.stg_fx_rates .................................. [RUN]
20:53:33  1 of 8 OK created sql view model demo.stg_fx_rates ............................. [OK in 0.03s]
20:53:33  2 of 8 START sql view model demo.stg_prices .................................... [RUN]
20:53:33  2 of 8 OK created sql view model demo.stg_prices ............................... [OK in 0.01s]
20:53:33  3 of 8 START seed file demo.fx_rates_raw ....................................... [RUN]
20:53:33  3 of 8 OK loaded seed file demo.fx_rates_raw ................................... [INSERT 9 in 0.02s]
20:53:33  4 of 8 START seed file demo.vendor_prices_raw .................................. [RUN]
20:53:34  4 of 8 OK loaded seed file demo.vendor_prices_raw ............

In [82]:
!dbt show --inline "select instrument_id, price_date, native_ccy, price_native, fx_rate_to_usd, price_usd from demo.fct_prices_usd where native_ccy = 'GBP' order by price_date" --limit 10
# One line changed in staging: the currency is now upper case. Build re-created the models and re-ran the tests; both pass, and the three rows have a dollar price.

20:53:35  Running with dbt=1.11.8
20:53:35  Registered adapter: duckdb=1.11.1
20:53:35  Found 2 seeds, 4 models, 2 data tests, 1 sql operation, 2 sources, 511 macros
20:53:35  
20:53:35  Concurrency: 1 threads (target='dev')
20:53:35  
Previewing inline node:
| instrument_id | price_date | native_ccy | price_native | fx_rate_to_usd | price_usd |
| ------------- | ---------- | ---------- | ------------ | -------------- | --------- |
| GB0005405286  | 2026-11-03 | GBP        |       6.988… |         1.267… |    8.854… |
| GB0005405286  | 2026-11-04 | GBP        |       7.256… |         1.268… |    9.201… |
| GB0005405286  | 2026-11-05 | GBP        |       7.134… |         1.268… |    9.045… |



<img src="../common/diagrams/png/d3-build-6.png" alt="Where we are: one line fixed in staging, and both tests pass." style="max-height:300px; max-width:100%;">

*Where we are: one line fixed in staging, and both tests pass.*

#### Step 7 — A test of your own: any query that returns the rows that break a rule

In [83]:
%%writefile tests/assert_prices_are_positive.sql
-- A test of your own: a query that returns the rows that break the rule. No rows, the test passes.
-- Rule: a price is never zero or negative.
select instrument_id, price_date, price_usd
from {{ ref('fct_prices_usd') }}
where price_usd <= 0

Writing tests/assert_prices_are_positive.sql


In [84]:
!dbt test --select assert_prices_are_positive
# A SQL file in tests/ is a test. This one passes: no price is zero or negative. If one ever is, the build stops here.

20:53:36  Running with dbt=1.11.8
20:53:36  Registered adapter: duckdb=1.11.1
20:53:36  Found 2 seeds, 4 models, 3 data tests, 2 sources, 511 macros
20:53:36  
20:53:36  Concurrency: 1 threads (target='dev')
20:53:36  
20:53:37  1 of 1 START test assert_prices_are_positive ................................... [RUN]
20:53:37  1 of 1 PASS assert_prices_are_positive ......................................... [PASS in 0.02s]
20:53:37  
20:53:37  Finished running 1 test in 0 hours 0 minutes and 0.09 seconds (0.09s).
20:53:37  
20:53:37  Completed successfully
20:53:37  
20:53:37  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


<img src="../common/diagrams/png/d3-build-7.png" alt="Where we are: four models, three tests, all passing." style="max-height:300px; max-width:100%;">

*Where we are: four models, three tests, all passing.*

#### Step 8 — Docs and lineage come from the same files

In [ ]:
# !dbt docs generate

In [ ]:
# !dbt docs serve --port 8081

In [ ]:
# !dbt ls --resource-type model test --select +fct_prices_usd
# # To browse the site, in a terminal in this folder: dbt docs serve --host 127.0.0.1 --port 8082   (the lineage graph is the blue button, bottom right)
# # The descriptions in schema.yml, the columns from the warehouse and the graph from ref() become a website. Nothing extra was written for it.

![The life cycle of a dbt model, as you just saw it](../common/diagrams/png/d3-lifecycle.png)

*The life cycle of a dbt model, as you just saw it*

<details><summary>Notes</summary>

- Declare the source, write the SELECT, run it, look at the rows.
- Write the promise as a test. When it fails, the fix goes in the model, never in the data.
- Build, document, commit: the pull request carries the SQL, the tests and the descriptions together.

</details>

<img src="../common/diagrams/png/d3-demo-to-challenge.png" alt="From this demo to the challenge: the same kinds of file, more of them" style="max-height:520px; max-width:100%;">

*From this demo to the challenge: the same kinds of file, more of them*

<a id="s2"></a>

## 2. The dbt practical: launch, Phase 1 and the walkthrough

### Launch — the pipeline you will build today

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">You have seen one model through its life cycle. Can you build a pipeline of ten, and tell when it is wrong?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>dbt from the inside: sources, staging, intermediate, marts, tests, docs, on the Portfolio Analytics case, on Databricks.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Reviewing generated code needs the same knowledge as writing it; the practical builds that knowledge one section at a time.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Work the README in order: concept, task, hint, check, variation, solution. Phase 1 by about 11:00, Phase 2 from there and after lunch.</div></div></div>

![The Portfolio Analytics pipeline the learners build: five sources, staging, intermediate, three marts, tests](../common/diagrams/png/d3-pipeline.png)

*The Portfolio Analytics pipeline the learners build: five sources, staging, intermediate, three marts, tests*

<details><summary>Notes</summary>

- Six raw feeds ship as seeds; five are modelled; refdata is there to inspect and for freshness practice.
- Staging renames and casts, one model per source; intermediate does the joins; marts are the consumer-aligned facts and aggregates.
- The tests are promises from a contract made executable: not null, unique, a relationship, accepted values, NAV within one basis point, the rollup reconciles.

</details>

### Challenge 9 — dbt practical: Phase 1 first

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; dbt practical (challenge 02) &nbsp;·&nbsp; about 3 hours across the day</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Build, test and document the Portfolio Analytics pipeline: eight staging sections, then ten challenges. &nbsp;·&nbsp; <code>challenges/v1/02-dbt-pipeline/README.md</code></div></div>

**Where to work**

1. `cd challenges/v1/02-dbt-pipeline/dbt`; every `dbt` command runs from here.
2. `dbt debug` (connection), `dbt seed` (eight CSVs into `training_dev.training_<you>`), `dbt ls --resource-type model` (ten stubs).
3. `mise run challenge:verify 02` and `challenge:reset 02` run from anywhere; never reset to "start clean": it deletes authored work.

**Phase 1, about an hour**

1. S1 Inspect the raw data.
2. S2 Read the dbt project.
3. S3 Declare sources and freshness.
4. S4 First staging model: `stg_holdings`.
5. S5 The remaining staging models.
6. S6 Materialisations: view vs table.
7. S7 Build the whole staging layer.
8. S8 A staging test.

**Phase 2**

1. C1 The enrichment join: `int_holdings_enriched`.
2. C2 The two supporting intermediate models.
3. C3 First curated mart: `agg_fund_exposure`.
4. C4 Second curated mart: `fct_portfolio_returns`.
5. C5 Generic tests on the marts.
6. C6 A singular reconciliation test: NAV within tolerance.
7. C7 The full build: `dbt build`.
8. C8 Docs and the lineage graph.
9. C9 The Databricks post-hook: `OPTIMIZE … ZORDER`.
10. C10 Capstone: a new mart end to end.

**Gate.** `mise run challenge:verify 02` runs seeds, models and tests; green means solved.

**Support.**

- Keep a list of commands, configuration changes, questions and discoveries; the recap is built from it.

### Walkthrough — Phase 1: sources and staging

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the same project, section by section</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The instructor's copy of the challenge 02 solution in <code>demos/portfolio_analytics/</code>: one cell per README section, for "what should I see?".</div></div>

1. Same model names, layers, tags and tests; the Databricks-only post-hook is a no-op macro here.
2. Use a cell when someone asks what they should see; `demos/portfolio_analytics/README.md` maps each cell to its README section.

In [ ]:
%cd {HERE}/demos/portfolio_analytics
# The walkthrough cells run in the instructor's copy of the challenge project.

#### S1 — Inspect the raw data: what feeds the models

In [ ]:
!dbt seed --full-refresh 

In [ ]:
sql("select * from training_demo.holdings_raw order by fund_cd, pos_dt limit 8", PORTFOLIO)
# A holding: one fund, one instrument, one day, valued in its local currency. Note fund_cd, pos_dt, isin, qty, mkt_val_lc, ccy_cd and _extracted_at.

#### S2 — Read the project: layers, materialisations, tags

In [ ]:
!dbt ls --resource-type model --exclude tag:stretch

In [ ]:
!cat dbt_project.yml
# Ten models in three layers. Layers are folders with +materialized and +tags: select a layer with --select tag:staging.

#### S3 — Sources and freshness: five lines per source table, because the fund master has no load timestamp

In [ ]:
!dbt source freshness
# Five sources report; product_ref has no loaded_at_field and is skipped. Tighten error_after to 0 hours and it flips to error: the check working, not a bug.

#### S4 — The first staging model: rename, cast, coalesce a null ISIN into CASH_<ccy>, no joins

In [ ]:
!cat models/staging/stg_holdings.sql

In [ ]:
!dbt build --select stg_holdings

In [ ]:
sql("select fund_id, position_date, instrument_id, quantity, mv_local_ccy, local_ccy from training_demo.stg_holdings order by 1,2 limit 8", PORTFOLIO)
# Same grain as the source. instrument_id is an ISIN or CASH_<ccy>, never null; quantities are typed decimals.

#### S5 to S7 — The remaining staging models, then the whole layer by tag

In [ ]:
!dbt build --select tag:staging

In [ ]:
sql("select * from training_demo.stg_prices order by instrument_id, price_date limit 8", PORTFOLIO)
# stg_positions and stg_prices dedup to the latest extract per natural key with row_number(); staging never raises the row count.

#### S8 — A staging test is a query that must return no rows

In [ ]:
!cat models/staging/schema.yml

In [ ]:
!dbt test --select stg_holdings
# not_null and unique compile to select-the-violations queries. Zero rows is a pass.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Walkthrough — Phase 2: intermediate, marts, tests, docs

#### C1 — The enrichment join: holdings × prices × FX, grain unchanged

In [ ]:
!cat models/intermediate/int_holdings_enriched.sql
# Left joins on (instrument, date) and (currency, GBP, date). Ask the room: what happens when the FX rate is missing? Here it defaults to 1.0 via coalesce; the running example on Day 2 chose to leave it null and fail the test. Both are decisions; the contract should say which.

#### C3, C4 — The marts: an aggregate by strategy and the daily-grain fact with a calculated NAV

In [ ]:
!dbt build --select +agg_fund_exposure +fct_portfolio_returns

In [ ]:
sql("select fund_id, position_date, strategy_code, exposure_base_ccy, instrument_count from training_demo.agg_fund_exposure order by 1,2", PORTFOLIO)

In [ ]:
sql("select fund_id, position_date, instrument_id, mv_local_ccy, fx_rate_to_base, mv_base_ccy, round(calc_nav_per_unit, 6) as calc_nav_per_unit from training_demo.fct_portfolio_returns order by 1,2", PORTFOLIO)
# calc_nav_per_unit = total base-currency market value / 100,000,000 synthetic units. Holdings only; cash excluded; a teaching measure, not an official NAV.

#### C5, C6 — Generic tests on the marts, and a singular reconciliation: NAV within one basis point of the custodian

In [ ]:
!cat tests/assert_nav_within_tolerance.sql

In [ ]:
sql("select c.fund_id, c.position_date, round(c.calc_nav_per_unit,4) as calc, cu.nav_per_unit as custodian, round((c.calc_nav_per_unit - cu.nav_per_unit)/cu.nav_per_unit*10000, 3) as variance_bps from training_demo.fct_portfolio_returns c join training_demo.seed_custodian_nav cu using (fund_id, position_date) order by 1,2", PORTFOLIO)
# The seeds are built so this passes within 1bp. Change one holding value and it will not: that is the test earning its keep.

#### C7, C8 — The full build, then docs and lineage

In [ ]:
!dbt build --exclude tag:stretch

In [ ]:
!dbt docs generate
print("To browse the site, in a terminal in this folder: dbt docs serve --host 127.0.0.1 --port 8081   (the --host matters: DBT_HOST is used for the Databricks host in the learners' profile)")

#### C9, C10 — The Databricks post-hook and the capstone mart

In [ ]:
!cat macros/optimize_mart.sql
# On Databricks: OPTIMIZE ... ZORDER BY (fund_id, position_date) after each mart, a Delta optimisation. Here: a no-op. A macro that dispatches on target.type is how one project runs on two warehouses.

In [ ]:
sql("select * from training_demo.agg_fund_daily_exposure order by 1,2", PORTFOLIO)

In [ ]:
!cat tests/reconcile_daily_exposure.sql
# C10: the (fund, date) rollup must preserve the strategy-level total. Learners create both the mart and this test.

### Circulating — Phase 2 to lunch: three questions per learner, and the symptom table

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 CIRCULATION CHECKLIST — three questions per learner</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Ask: could this join pass its tests and still double the reported value? A successful build alone is insufficient.</div></div>

- Sources and staging: can they say what one row means, name its key, and explain the types and units?
- Joins and models: are the lookup keys unique on the right-hand side, do dates belong in the join, and what happens to unmatched rows?
- Tests and output: can they show a value checked independently of the model that produced it?
- Check the row grain, right-side uniqueness, unmatched rows and an independent total.

| Symptom | Cause | Fix |
|---|---|---|
| `auth_type: oauth is required when not using access token` | The shell has no `DBT_ACCESS_TOKEN`; mise did not load the environment | Open a fresh terminal, or `mise run setup:databricks`; do not re-authenticate blindly (the repo's Copilot instructions give a three-step check) |
| `dbt show --inline "select …"` errors with unexpected extra arguments | Running through `mise run challenge:dbt` word-splits the SQL | Run raw `dbt` from the project folder, or use `dbt show --select <model>` |
| `--select tag:staging` selects nothing | The `models:` block in `dbt_project.yml` is missing or mis-keyed | Compare with the shipped `dbt_project.yml`; the project name must match |
| Full build fails in C7 on a placeholder mart | The capstone mart is created in C10, not shipped | Build with `--exclude agg_fund_daily_exposure` until C10 |
| Python crashes on the first query that returns rows | pandas result conversion on this image | The shipped profile sets `_disable_pandas: true`; check it is still there |
| `dbt docs serve` cannot bind | `DBT_HOST` holds the Databricks host | `dbt docs serve --host 127.0.0.1 --port 8081` |
| A relationship test fails after a learner edits a key | Keys changed in staging without the fund master | Trace the key back to the source; staging must not invent keys |

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s3"></a>

## 3. Advanced dbt, then complete and extend the practical

### Demo — Jinja that writes SQL: a loop and a macro

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">This morning's Jinja filled in names. What else can it do?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A loop and a macro on this morning's price sheet, then what OPTIMIZE and ZORDER do on Databricks. Generic tests of your own, snapshots, incremental models and unit tests are yours to explore.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>These are the pieces the platform's vended project uses, and the macros you will read in Friday's capstone.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>A short demo on screen. Then carry on with Phase 2, or ask Copilot to add one of these to your project.</div></div></div>

![A macro: SQL written once, expanded wherever it is called](../common/diagrams/png/d3-macro.png)

*A macro: SQL written once, expanded wherever it is called*

<details><summary>Notes</summary>

- A macro lives in `macros/` and takes arguments, like a function. It returns SQL text.
- Every model that calls it gets the same expression at compile time; change the macro and every model changes.
- The warehouse never sees the macro, only the expanded SQL in `target/compiled/`.

</details>

In [85]:
%cd {HERE}/demos/price_sheet
# Back in this morning's price sheet project.

/Users/usama.shahid/Library/CloudStorage/GoogleDrive-usama.shahid@decoded.com/My Drive/cg-dxp/courseware/day3-dbt/demos/price_sheet


#### A loop: four columns from one line

In [86]:
%%writefile models/marts/agg_prices_by_ccy.sql
-- agg_prices_by_ccy: one column per currency, written by a loop instead of by hand
{% set currencies = ['USD', 'EUR', 'GBP', 'JPY'] %}

select 
    price_date,
    {% for ccy in currencies %}
    count(case when native_ccy = '{{ ccy }}' then 1 end) as prices_in_{{ ccy | lower }}{% if not loop.last %},{% endif %}
    {% endfor %}
from {{ ref('fct_prices_usd') }}
group by price_date
order by price_date

Writing models/marts/agg_prices_by_ccy.sql


In [87]:
!dbt show --select agg_prices_by_ccy

22:31:20  Running with dbt=1.11.8
22:31:20  Registered adapter: duckdb=1.11.1
22:31:20  Found 2 seeds, 5 models, 3 data tests, 2 sources, 511 macros
22:31:20  
22:31:20  Concurrency: 1 threads (target='dev')
22:31:20  
Previewing node 'agg_prices_by_ccy':
| price_date | prices_in_usd | prices_in_eur | prices_in_gbp | prices_in_jpy |
| ---------- | ------------- | ------------- | ------------- | ------------- |
| 2026-11-03 |             9 |             1 |             1 |             1 |
| 2026-11-04 |            10 |             1 |             1 |             1 |
| 2026-11-05 |             9 |             0 |             1 |             1 |



In [88]:
!cat target/compiled/price_sheet/models/marts/agg_prices_by_ccy.sql
# The loop ran at compile time and wrote four count(...) lines. Add a currency to the list and the model gains a column.

-- agg_prices_by_ccy: one column per currency, written by a loop instead of by hand


select 
    price_date,
    
    count(case when native_ccy = 'USD' then 1 end) as prices_in_usd,
    
    count(case when native_ccy = 'EUR' then 1 end) as prices_in_eur,
    
    count(case when native_ccy = 'GBP' then 1 end) as prices_in_gbp,
    
    count(case when native_ccy = 'JPY' then 1 end) as prices_in_jpy
    
from "price_sheet"."demo"."fct_prices_usd"
group by price_date
order by price_date

#### A macro: the conversion rule in one place

In [90]:
%%writefile macros/to_usd.sql
{# to_usd: the conversion rule, written once. A macro is a function that returns SQL text. #}
{% macro to_usd(amount, ccy, rate) %}
    cast({{ amount }} * case when {{ ccy }} = 'USD' then 1.0 else {{ rate }} end as decimal(18, 6))
{% endmacro %}

Overwriting macros/to_usd.sql


In [91]:
%%writefile models/intermediate/int_prices_usd.sql
-- int_prices_usd: every price joined to its day's FX rate and converted. Same rows as stg_prices.
select
    p.instrument_id,
    p.price_date,
    p.native_ccy,
    p.price_native,
    case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end   as fx_rate_to_usd,
    {{ to_usd('p.price_native', 'p.native_ccy', 'f.fx_rate') }}  as price_usd
from {{ ref('stg_prices') }} p
left join {{ ref('stg_fx_rates') }} f
    on  f.from_ccy  = p.native_ccy
    and f.rate_date = p.price_date

Overwriting models/intermediate/int_prices_usd.sql


In [ ]:
!dbt build --select int_prices_usd+
# The + after the name: this model and everything downstream of it (fct_prices_usd, agg_prices_by_ccy) and their tests. Before the name, +int_prices_usd, it would be everything upstream.

22:34:20  Running with dbt=1.11.8
22:34:20  Registered adapter: duckdb=1.11.1
22:34:21  Found 2 seeds, 5 models, 3 data tests, 2 sources, 512 macros
22:34:21  
22:34:21  Concurrency: 1 threads (target='dev')
22:34:21  
22:34:21  1 of 6 START sql view model demo.int_prices_usd ................................ [RUN]
22:34:21  1 of 6 OK created sql view model demo.int_prices_usd ........................... [OK in 0.03s]
22:34:21  2 of 6 START sql table model demo.fct_prices_usd ............................... [RUN]
22:34:21  2 of 6 OK created sql table model demo.fct_prices_usd .......................... [OK in 0.03s]
22:34:21  3 of 6 START test assert_prices_are_positive ................................... [RUN]
22:34:21  3 of 6 PASS assert_prices_are_positive ......................................... [PASS in 0.02s]
22:34:21  4 of 6 START test not_null_fct_prices_usd_instrument_id ........................ [RUN]
22:34:21  4 of 6 PASS not_null_fct_prices_usd_instrument_id ................

In [93]:
!cat target/compiled/price_sheet/models/intermediate/int_prices_usd.sql
# The model now calls to_usd(...); the compiled SQL shows the expression the macro returned. The tests still pass: same answer, one definition.

-- int_prices_usd: every price joined to its day's FX rate and converted. Same rows as stg_prices.
select
    p.instrument_id,
    p.price_date,
    p.native_ccy,
    p.price_native,
    case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end   as fx_rate_to_usd,
    
    cast(p.price_native * case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end as decimal(18, 6))
  as price_usd
from "price_sheet"."demo"."stg_prices" p
left join "price_sheet"."demo"."stg_fx_rates" f
    on  f.from_ccy  = p.native_ccy
    and f.rate_date = p.price_date

<details><summary><strong>dbt docs: Jinja and macros</strong></summary>

- [Jinja and macros](https://docs.getdbt.com/docs/build/jinja-macros): what Jinja does in dbt, and how to write and call a macro.
- [Using Jinja](https://docs.getdbt.com/guides/using-jinja): a step-by-step guide, from a loop in one model to a macro you reuse.
- [How we style our Jinja](https://docs.getdbt.com/best-practices/how-we-style/4-how-we-style-our-jinja): dbt Labs' guideline for readable Jinja, and when not to use a macro.
- [dbt Jinja functions](https://docs.getdbt.com/reference/dbt-jinja-functions-context-variables): everything dbt adds to Jinja: `ref`, `source`, `config`, `this`, `var`, `is_incremental`.
- [Graph operators](https://docs.getdbt.com/reference/node-selection/graph-operators): the `+` in `--select int_prices_usd+`.
- [Jinja template designer](https://jinja.palletsprojects.com/en/stable/templates/): the Jinja language itself: `if`, `for`, `set`, filters such as `lower`.

</details>

<details><summary><strong>On your own: generic tests, snapshots, incremental models and unit tests</strong></summary>

- **A generic test of your own** ([docs](https://docs.getdbt.com/best-practices/writing-custom-generic-tests))**:** a test with arguments, in `tests/generic/`, used in the YAML like `not_null`. This morning's "prices are positive" rule could become one.
- **Snapshot** ([docs](https://docs.getdbt.com/docs/build/snapshots))**:** keeps every version of a row that changes, with the dates each version was true. `dbt snapshot` runs it.
- **Incremental model** ([docs](https://docs.getdbt.com/docs/build/incremental-models))**:** a table that adds only the new rows on each run instead of rebuilding. The filter sits inside `{% if is_incremental() %}`.
- **Unit test** ([docs](https://docs.getdbt.com/docs/build/unit-tests))**:** runs a model's SQL on made-up rows and compares the result with the rows you expect, so the logic is checked without warehouse data. To try one, ask Copilot for a unit test of `stg_holdings` in your project.

Snapshots and incremental models are in `common/glossaries/dbt-commands.md`, under materialisations. To try one, ask Copilot to add a snapshot of the fund master or an incremental version of a mart to your project, run it twice, and read what changed.

</details>

#### On Databricks: OPTIMIZE and ZORDER, the post-hook in challenge C9

- Optimise: combining small files into larger files
    -- WHY do it ?
    -- it's parquet format - small would take more space 
    -- read i/o would be optimised if we less amount of files

- Zorder is sorting
- we organised by columns
- sorted data via values into files

-- Root cause of creating alot of small parquet files
 -- partioning based on columns
 -- When a new transacion comes in databricks - when we ask for a certain change (UPDATEM, DELETE),
 databricks locates the parquet file which contains the specfic data - and the

```yaml
# dbt_project.yml in the challenge: after dbt builds each mart, Databricks runs this
    marts:
      +materialized: table
      +post-hook: "OPTIMIZE {{ this }} ZORDER BY (fund_id, position_date)"
```

![OPTIMIZE: many small files become a few large ones. Same rows, faster to read.](../common/diagrams/png/d3-optimize.png)

*OPTIMIZE: many small files become a few large ones. Same rows, faster to read.*

<details><summary>Notes</summary>

- A Delta table is a folder of Parquet files, and every write adds more, often small ones. Opening thousands of small files is slow.
- `OPTIMIZE` rewrites them into fewer, larger files. The data does not change; the old files stay until `VACUUM`, so time travel still works.
- It costs compute: fine after every build for a small mart, usually scheduled for a large table.

</details>

![ZORDER BY: rows with similar values share a file, so a query can skip the files it does not need](../common/diagrams/png/d3-zorder.png)

*ZORDER BY: rows with similar values share a file, so a query can skip the files it does not need*

<details><summary>Notes</summary>

- Delta records the smallest and largest value of each column in every file. A filter can skip any file whose range cannot match.
- `ZORDER BY (fund_id, position_date)` sorts the rows while compacting, so each file covers a narrow range of funds and dates.
- Choose the columns that queries filter and join on; two or three at most, because each extra column weakens the effect.
- These are Delta operations: they run on Databricks only. On the training data, a handful of rows, you will not see a difference; the hook shows the pattern.

</details>

In [ ]:
%cd {HERE}

### Challenge 9 continued — Phase 2, then your choice of stretch

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 14:20</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s4"></a>

## 4. Consolidation

### Activity 9b — second look at yesterday's AI-generated dbt

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 FIRST — two things you could not judge yesterday</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Yesterday afternoon Copilot and Catalyst generated a dbt project for you, and the room asked: do we trust it, review all of it, or review some of it? Write down two things in that project you could not judge.</div></div>

- A file or folder whose purpose you could not tell.
- SQL you could not say was right: a join, a calculation, the curly braces.
- A test you could not say was checking anything.
- Something that looked like more than you asked for.

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 9b &nbsp;—&nbsp; Second look at yesterday's AI-generated dbt &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Yesterday Copilot and Catalyst generated a dbt project from your contract, before you knew dbt. First, write down two things in that project you could not judge yesterday. Then reopen it with a partner (if yours did not finish, use your partner's) and assess it in graph order: the sources, then staging, then the joins, then the tests, then anything nobody asked for. Finally, pick one thing you caught and prompt Copilot again to fix it: name the file, say what is wrong and say how you will check the fix. Read what it changed and run the check.</strong><br><br><strong>Produce:</strong><br>• Two things you could not judge yesterday, each marked: can judge now, or still cannot<br>• One thing you catch now that you would have approved yesterday, with the file it is in<br>• The prompt you wrote to fix it, and whether the fix passed your check<br>• One thing you still cannot judge, for the questions table<br><br><strong>Debrief questions:</strong><br>• What did you catch today that you would have approved yesterday?<br>• How is today's prompt different from yesterday's?<br>• Did the AI write anything nobody asked for? What would you delete?</div></div>

**What to look for, in order**

| Where | Ask |
|---|---|
| `sources.yml` | Is every raw table declared, or are there hard-coded table names in the SQL? |
| Staging models | One row per what? Are names, types and codes cleaned here, and are duplicates handled? |
| Joins | Are the keys unique on the right-hand side? What happens to a row with no match: dropped, null, or a silent default? |
| Tests | Does each promise in your contract (keys, not null, accepted values, a reconciliation) have a test? Would any test fail if the data were wrong? |
| `dbt_project.yml`, `config()` | Which models are views and which are tables, and is that what consumers need? |
| Everything else | Which files, models or macros did nobody ask for? |

Open one model's file under `target/compiled/` if you are unsure what the warehouse receives.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, in yesterday's generated project</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">In models/&lt;the file&gt;, &lt;what is wrong: for example, the join to FX rates drops prices with no rate for that date&gt;.
Change only that model: &lt;what you want instead&gt;. Add or tighten a test in the schema YAML that would fail if this went wrong again.
Show me the diff, then run dbt build --select &lt;the model&gt;+ and show me the result.</pre></div></div>

### Debrief — what did we learn?

**🖊️ Whiteboard — what did we learn? (fill gaps from this list, do not lecture it)**

- Models and layers
- Dependencies
- Tests
- Views and tables
- Jinja and macros
- Reviewing generated code

### Activity 9 — test review in pairs

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 9 &nbsp;—&nbsp; Test review in pairs &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 10 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Pick one test from your project and show it to your partner. Together, name one wrong result that would still get past it. Agree one extra test that would catch it.</strong><br><br><strong>Produce:</strong><br>• One test, and one wrong result it would miss<br>• One extra test that would catch it<br><br><strong>Debrief questions:</strong><br>• Which wrong result would have passed every test in your project?</div></div>

### Close — closing discussion

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ Two things each learner could not judge in yesterday's generated project, marked as judged or still open; one thing they caught, and the prompt that fixed it.<br>☐ The working dbt project, model and test results, one independent output check and the outstanding questions for orchestration, kept where they are.<br>☐ Each learner can trace one plain-English mapping through its model and its test.<br>☐ The Airflow baseline is ready: Databricks auth works, challenge 02's seeds are loaded, and Astro has been started once after setup.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>